# Naive Bayes, Explained Simply
Classifying text by weighing each word as a clue. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A restaurant website has 400 short reviews, each labelled positive or negative.

### 400 reviews
- `reviews.csv`: 400 reviews with a sentiment: positive or negative
- `df["review"][:3]`: the first three reviews

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "naive-bayes/reviews.csv")
df = pd.read_csv(url)
print(df.shape)
print(df["sentiment"].value_counts().to_dict())
for r in df["review"][:3]:
    print("-", r)

## 2. Words into numbers
**Bag of words**: count how many times each word appears in each review, ignoring the order.

### Count the words
- `CountVectorizer(stop_words="english")`: turns text into word counts, skipping common words like 'the' and 'was'
- `fit_transform(X_train)`: learn the vocabulary from the training reviews, and count
- `transform(X_test)`: count the test reviews using the same vocabulary
- `the shape`: 300 reviews x one column per word

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["review"], df["sentiment"], test_size=0.25, random_state=0)
vec = CountVectorizer(stop_words="english")
X_train_counts = vec.fit_transform(X_train)
X_test_counts = vec.transform(X_test)
print("vocabulary size:", len(vec.vocabulary_))
print(X_train_counts.shape)

### One review as counts
- `vec.transform([...])`: count the words of a new sentence
- `example.indices`: which words appear
- `the output`: just the useful words, with their counts

In [ ]:
example = vec.transform(["The pizza was really tasty and the staff friendly"])
words = vec.get_feature_names_out()
print({words[i]: int(n) for i, n in zip(example.indices, example.data)})

## 3. Naive Bayes
For each class, **score = P(class) x P(word1 | class) x P(word2 | class) x ...**. The class with the bigger score wins. 'Naive' because it treats each word as an independent clue.

### Train and test
- `MultinomialNB`: Naive Bayes for word counts
- `alpha=1.0`: add-one smoothing: pretend every word was seen once more in each class
- `fit`: counts how often each word appears in positive and negative reviews: that's all the training is

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

model = MultinomialNB(alpha=1.0)
model.fit(X_train_counts, y_train)
pred = model.predict(X_test_counts)
print("test accuracy:", round(accuracy_score(y_test, pred), 3))

### The most telling words
- `feature_log_prob_`: the log of P(word | class), for every word and both classes
- `ratio`: how much more likely a word is in positive than negative reviews (on the log scale)
- `np.argsort`: sort the words by that ratio

In [ ]:
import numpy as np

log_p = model.feature_log_prob_
ratio = log_p[1] - log_p[0]
order = np.argsort(ratio)
print("model classes:", model.classes_)
print("most positive:", list(words[order[-8:]][::-1]))
print("most negative:", list(words[order[:8]]))

### Classify new reviews
- `predict_proba`: the probability of each class: column 1 is positive
- `the mixed review`: tasty versus slow: the clues pull both ways

In [ ]:
new = ["The coffee was amazing and the waiter friendly",
       "Cold soup and a rude waiter",
       "The pasta was tasty but the service was slow"]
probs = model.predict_proba(vec.transform(new))
for text, p in zip(new, probs):
    print(f"{p[1]:.0%} positive  |  {text}")

## 4. Why smoothing matters
Without smoothing, a word never seen in one class gets probability 0, and one zero wipes out the whole product. **Smoothing** adds alpha to every count.

### One unseen word ruins everything
- `tiny`: six tiny training reviews; 'cold' never appears in a positive one
- `review`: three 'great's and one 'cold'
- `alpha 1e-10`: almost no smoothing: P(cold | positive) = 0, so positive scores 0, whatever else the review says
- `alpha 1.0`: with add-one smoothing, the three 'great's win
- `force_alpha=True`: let us use a really tiny alpha

In [ ]:
tiny = ["great food", "great staff", "lovely food",
        "cold food", "rude staff", "great view but cold food"]
labels = ["positive"] * 3 + ["negative"] * 3
v = CountVectorizer().fit(tiny)
review = v.transform(["great great great food, slightly cold"])
for a in [1e-10, 1.0]:
    m = MultinomialNB(alpha=a, force_alpha=True).fit(v.transform(tiny), labels)
    print(f"alpha {a}: {m.predict_proba(review)[0][1]:.0%} positive")

## Try it yourself
1. Write your own review and classify it.
2. Try alpha=0.01 and alpha=10 in MultinomialNB. How does test accuracy change?
3. Print the 5 most negative words instead of the 8 most positive.